In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load Data

In [17]:
full_mic_data = pd.read_csv('MIC_data_cleaned.csv')
all_chem_phys = pd.read_csv('all_chem_phys_features.txt', sep='\t', index_col=0)
all_chem_phys.index.name = 'Peptide ID'
all_embeddings = pd.read_csv('all_embeddings.tsv', sep='\t', index_col=0)
all_embeddings.index.name = 'Peptide ID'


In [18]:
full_mic_data['Peptide ID'].nunique()

23771

In [19]:
all_chem_phys.index.nunique()

17601

In [20]:
all_embeddings.index.nunique()


17601

In [21]:
peptides_w_data = list(set(full_mic_data['Peptide ID']) & set(all_chem_phys.index))
len(peptides_w_data)

17294

## Ensure clean dataset

In [25]:
all_chem_phys.iloc[:, 1:-3]


,Normalized Hydrophobic Moment,Normalized Hydrophobicity,Net Charge,Isoelectric Point,Penetration Depth,Tilt Angle,Disordered Conformation Propensity,Linear Moment,Propensity to in vitro Aggregation,Angle Subtended by the Hydrophobic Residues,Amphiphilicity Index,Propensity to PPII coil
Peptide ID,,,,,,,,,,,,
8,1.97,-1.07,5.0,14.00,10,87,0.24,0.26,0.00,190.0,1.80,0.98
10,0.41,-3.25,1.0,14.00,0,62,1.25,0.00,500.03,360.0,0.00,0.96
11,1.08,0.07,8.0,11.79,13,86,-0.09,0.28,18.63,30.0,1.60,1.01
12,0.46,0.81,4.0,11.85,22,172,-0.27,0.39,0.00,50.0,1.17,1.02
14,1.01,-0.29,3.0,10.99,15,88,0.23,0.20,0.61,70.0,0.91,0.89
...,...,...,...,...,...,...,...,...,...,...,...,...
22722,2.12,-0.75,4.0,11.16,14,75,0.17,0.33,11.66,190.0,1.13,0.84
22723,2.14,-0.57,5.0,11.28,14,72,0.03,0.28,0.00,160.0,1.41,0.82
22724,1.89,-0.42,4.0,11.15,14,80,0.06,0.29,0.00,160.0,1.13,0.97


In [29]:
has_data_noDup = full_mic_data[full_mic_data['Peptide ID'].isin(peptides_w_data)].drop_duplicates(['Peptide ID', 'Target Species'])

In [38]:
# test data (only a few species, no duplicates)
top_species = has_data_noDup.groupby('Target Species').size().sort_values(ascending=False).iloc[:3].index
test_mic = has_data_noDup[has_data_noDup['Target Species'].isin(top_species)][['Peptide ID', 'Target Species', 'Activity']].dropna()
test_mic = test_mic[test_mic["Activity"].astype(str).str.match("^[0-9.]+$", na=False)]
test_mic['mic'] = test_mic['Activity'].astype(float)
test_mic.shape

(12445, 3)

In [ ]:
from pmf_model import Hybrid_PMF

hpmf_model = Hybrid_PMF(test_mic.iloc[::10].copy(), all_chem_phys.iloc[:, 1:-3].copy(), all_embeddings.copy(),
                        None, dim=10, include_esm=False)

hpmf_model.draw_samples(draws=300, tune=100)

In [ ]:
hpmf_model.var_inference()

In [ ]:
plt.plot(hpmf_model.idata_vi.hist)

In [ ]:
# Need to do prediction on test data